# Week 10, Day 2 Lab — Real APIs, Error Handling & Retries
### From fake data to the live internet, in 2.5 hours

**Yesterday** your agent called fake, hardcoded functions. **Today every tool hits a real,
live service** — a real weather API, a real Wikipedia search, and a real (local) database.
Real services fail in ways fake ones never do, so today is also about making your agent
survive that.

**How to use this notebook:**
- **WRITE THIS** cells have only a spec/signature — you write the body.
- **TODO** cells ask you to modify or extend something already there.
- Plain cells are given — run them as-is.
- Stuck 5+ minutes on a WRITE THIS cell? The **Appendix** at the very bottom has solutions.

**Total time: 150 minutes**, including one 10-minute break.


---
## Section 0 — Setup

Same as yesterday: Gemini 2.5 Flash via Google's free `google-genai` SDK.

If you don't already have a key from Day 1: go to **https://aistudio.google.com/apikey**,
sign in, click **Create API key**, copy it. (If your Colab runtime restarted, you'll need to
re-enter it even if you used one yesterday — nothing is saved between sessions.)


In [1]:
!pip install -q google-genai


In [2]:
import os
import json
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [3]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.6-flash"

test = client.models.generate_content(model=MODEL, contents="Say 'Day 2 lab is ready!'")
print(test.text)


Day 2 lab is ready!


---
## Section 1 — Recap: Fake vs. Real (15 minutes)

Yesterday's `get_weather` looked like this:

```python
fake_data = {"lahore": {"forecast": "sunny", "temp_c": 34}, ...}
```

Today, let's hit an **actual live weather API** directly — no agent, no tool schema yet,
just a plain HTTP request — so you can see what "real" costs us that "fake" didn't.


In [4]:
import requests

# Open-Meteo is a free, real weather API that needs NO API key at all.
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 31.5497, "longitude": 74.3436, "current_weather": True},
    timeout=5
)
response.raise_for_status()
print(response.json()["current_weather"])


{'time': '2026-09-05T08:30', 'interval': 900, 'temperature': 32.8, 'windspeed': 2.5, 'winddirection': 249, 'is_day': 1, 'weathercode': 0}


In [5]:
response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
    "latitude": 24.8607,
    "longitude": 67.0011,
    "current_weather": True
},  # <-- change
    timeout=5
)
response.raise_for_status()
print(response.json()["current_weather"])


{'time': '2026-09-05T08:45', 'interval': 900, 'temperature': 29.2, 'windspeed': 18.2, 'winddirection': 244, 'is_day': 1, 'weathercode': 51}


---
## Section 2 — Worked Example: A Real Weather Tool, End to End

Here's the answer to "what about any city name?" — Open-Meteo also has a free **geocoding**
API (also no key needed) that turns a city name into coordinates. Our real tool will chain
**two** live API calls together: geocode the city, then fetch its weather.

This is fully worked for you — read it carefully, since Sections 4–5 ask you to write tools
with this exact same shape yourself.


In [6]:
# GIVEN — the real tool function: two chained live API calls
def get_weather(city: str) -> dict:
    """Look up real, current weather for any city name using free Open-Meteo APIs."""

    # Find the coordinates for this city
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=5
    )

    geo.raise_for_status() # gets the search results.
    # If nothing is found. Instead of crashing, the function returns an error dictionary
    results = geo.json().get("results")
    if not results:
        return {"error": f"Could not find a location matching '{city}'"}

    # Extract coordinates
    lat, lon = results[0]["latitude"], results[0]["longitude"]
    resolved_name = results[0]["name"]


    # Second API call: get weather
    # This sends those coordinates to the weather API and asks for current weather.
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": True},
        timeout=5
    )
    weather.raise_for_status() # extracts the weather information.
    current = weather.json()["current_weather"]
    return {"city": resolved_name, "temp_c": current["temperature"], "windspeed": current["windspeed"]} # returns only the information your agent needs.


# Quick manual tests — try a real city and a nonsense one
print(get_weather("Lahore"))
print(get_weather("Xyzzyxutopia"))   # should return the {"error": ...} branch, not crash


{'city': 'Lahore', 'temp_c': 32.8, 'windspeed': 1.8}
{'error': "Could not find a location matching 'Xyzzyxutopia'"}


In [7]:
# GIVEN — the schema
# This part doesn't perform the weather search.It tells Gemini, "I have a tool called get_weather. Give it a city name when you need weather."
get_weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get real, current weather for any city name in the world.",
    parameters={
        "type": "object",
        "properties": {"city": {"type": "string", "description": "Any city name, e.g. 'Lahore' or 'Boston'"}},
        "required": ["city"]
    }
)

weather_tool = types.Tool(function_declarations=[get_weather_declaration])
config = types.GenerateContentConfig(tools=[weather_tool])
tool_registry = {"get_weather": get_weather}


### Bring back your `run_agent` loop from Day 1

Paste in the `run_agent` function you built yesterday (or use the version below if you'd
rather start from a clean copy — it's identical to the Day 1 Appendix solution).


In [8]:
def run_agent(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            if verbose:
                print(f"[Step {step+1}] Thought: I have enough info. Giving final answer.")
            return response.text

        function_response_parts = []
        for fc in function_calls:
            name = fc.name
            args = dict(fc.args)
            if verbose:
                print(f"[Step {step+1}] Action: {name}({args})")
            result = tool_registry[name](**args)
            if verbose:
                print(f"[Step {step+1}] Observation: {result}")
            function_response_parts.append(
                types.Part.from_function_response(name=name, response={"result": result})
            )

        response = chat.send_message(function_response_parts)

    return "Reached max steps without a final answer."


In [9]:
answer = run_agent("What's the weather like in Tokyo right now?", config=config, tool_registry=tool_registry)
print()
print("=" * 50)
print("FINAL ANSWER:", answer)


[Step 1] Action: get_weather({'city': 'Tokyo'})
[Step 1] Observation: {'city': 'Tokyo', 'temp_c': 24.3, 'windspeed': 4.0}
[Step 2] Thought: I have enough info. Giving final answer.

FINAL ANSWER: The current weather in Tokyo is approximately 24.3°C (75.7°F) with a wind speed of 4 km/h.


**Notice:** this used yesterday's exact loop, unmodified — only the tool underneath changed
from fake to real. That's the whole point of building a generic loop: the tools are swappable.

**🔧 TODO Checkpoint (5 min):** ask about a city you're confident doesn't exist (misspell one
badly) and confirm your agent handles the `{"error": ...}` gracefully in its final answer,
rather than crashing.


In [10]:

answer = run_agent(
    "What's the weather in Xyzzyxutopia?",
    config=config,
    tool_registry=tool_registry
)

print(answer)

[Step 1] Action: get_weather({'city': 'Xyzzyxutopia'})
[Step 1] Observation: {'error': "Could not find a location matching 'Xyzzyxutopia'"}
[Step 2] Thought: I have enough info. Giving final answer.
I couldn't find a location named **Xyzzyxutopia**. It appears to be a fictional or unrecognized place. 

If you meant a different real-world city, please let me know and I'd be happy to check the weather for you!


---
## Section 3 — Error Handling & Retries (25 minutes)

Real APIs time out, rate-limit you, or occasionally the model passes an argument that doesn't
match what your function expects. Right now, if `get_weather` throws an uncaught exception
(e.g. a network timeout), your entire `run_agent` call crashes. Let's fix that at two levels.

### Level 1 — Safe argument handling

If the model ever calls a tool with the wrong argument names/types, `tool_registry[name](**args)`
raises a `TypeError` that currently kills the whole loop. Wrap it.


In [ ]:
# Error Handling: Catch an error so the program doesn't crash
# Retry: Try the operation again when a temporary failure occurs.
# Exceptional Backoff: Wait progressively longer between retries: 1s → 2s → 4s → 8s...

In [11]:
# safe_call_tool = Don't let a tool error crash my agent
def safe_call_tool(name, args, tool_registry):
    try:
        return tool_registry[name](**args) # Try to execute the tool

    except TypeError as e:
        return {
            "error": f"Invalid arguments for {name}: {e}" # If the tool fails because the arguments are wrong, catch that error."
        }

    except Exception as e:
        return {
            "error": f"Unexpected error in {name}: {e}" # If something else goes wrong, catch that too.The "e" contains the actual error message.
        }

# Test: this should print an error dict, NOT raise an exception
print(safe_call_tool("get_weather", {"wrong_argument_name": "Lahore"}, tool_registry))


{'error': "Invalid arguments for get_weather: get_weather() got an unexpected keyword argument 'wrong_argument_name'"}


### Level 2 — Retry with exponential backoff

Network calls fail transiently sometimes — a retry a second later often just works. Write the
retry wrapper from the spec below


In [12]:
# call_with_retry, protects against temporary network errors:
import time

def call_with_retry(func, max_retries=3, **kwargs):
    for attempt in range(max_retries):

        try:
            return func(**kwargs) # Try calling the function

        except requests.exceptions.RequestException as e: # If the function raises a network-related RequestException, we catch it

            # Last attempt failed
            if attempt == max_retries - 1: # it checks "Was this our final attempt?"
            # If yes:
                return {
                    "error": f"Failed after {max_retries} attempts: {e}"
                }
            # Otherwise:
            # Wait before trying again
            time.sleep(2 ** attempt)

# Test with a deliberately broken URL-based function to confirm retries + eventual failure message
def flaky_call():
    return requests.get("https://thisdomaindefinitelydoesnotexist12345.com", timeout=2).json()

print(call_with_retry(flaky_call, max_retries=2))


{'error': 'Failed after 2 attempts: HTTPSConnectionPool(host=\'thisdomaindefinitelydoesnotexist12345.com\', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f5791a6ccd0>: Failed to resolve \'thisdomaindefinitelydoesnotexist12345.com\' ([Errno -2] Name or service not known)"))'}


### Harden `run_agent` with both

**🔧 TODO:** modify the loop below (a copy of your `run_agent`) so that instead of calling
`tool_registry[name](**args)` directly, it uses your two new functions together — retry the
safe call. One line changes; the rest of the loop stays the same.


In [13]:
def run_agent_v2(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):
        parts = response.candidates[0].content.parts
        function_calls = [p.function_call for p in parts if p.function_call]

        if not function_calls:
            if verbose:
                print(f"[Step {step+1}] Thought: I have enough info. Giving final answer.")
            return response.text

        function_response_parts = []

        for fc in function_calls:
            name = fc.name
            args = dict(fc.args)

            if verbose:
                print(f"[Step {step+1}] Action: {name}({args})")

            # Safe tool call + retry wrapper
            result = call_with_retry(
                lambda: safe_call_tool(name, args, tool_registry)
            )

            if verbose:
                print(f"[Step {step+1}] Observation: {result}")

            function_response_parts.append(
                types.Part.from_function_response(
                    name=name,
                    response={"result": result}
                )
            )

        response = chat.send_message(function_response_parts)

    return "Reached max steps without a final answer."

In [14]:
# Test your hardened loop — should behave identically to run_agent for normal queries
answer = run_agent_v2("What's the weather in Lahore?", config=config, tool_registry=tool_registry)
print(answer)


[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 32.8, 'windspeed': 1.8}
[Step 2] Thought: I have enough info. Giving final answer.
The current weather in Lahore is 32.8°C with a wind speed of 1.8 km/h.


---
## Section 4 — Write a Real Search Tool: Wikipedia

Your turn to build a real-API tool completely from a spec, following the exact shape of
`get_weather` in Section 2. We'll use Wikipedia's public search API — real, free, no key.

**API reference (given):**
```
GET https://en.wikipedia.org/w/api.php
params: {"action": "query", "list": "search", "srsearch": <your query>,
         "format": "json", "srlimit": <num_results>}
```
A successful response looks like:
```json
{"query": {"search": [
    {"title": "Lahore", "snippet": "Lahore is the capital of..."},
    {"title": "Lahore Fort", "snippet": "..."}
]}}
```

**Spec for the function:**
- Signature: `search_wikipedia(query: str, num_results: int = 3) -> dict`
- Call the API above with `timeout=5` and `resp.raise_for_status()`
- Return `{"results": [{"title": ..., "snippet": ...}, ...]}` using the top `num_results` hits
- If there are zero results, return `{"results": []}` (not an error — an empty result is valid)


In [15]:
def search_wikipedia(query: str, num_results: int = 3) -> dict:
    headers = {
        "User-Agent": "AI-Agent-Lab/1.0 (educational project)"
    }

    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "format": "json",
            "srlimit": num_results
        },
        headers=headers,
        timeout=5
    )

    response.raise_for_status()

    search_results = response.json().get("query", {}).get("search", [])

    results = []

    for item in search_results[:num_results]:
        results.append({
            "title": item["title"],
            "snippet": item["snippet"]
        })

    return {
        "results": results
    }


# Test
print(search_wikipedia("Lahore Fort"))

{'results': [{'title': 'Lahore Fort', 'snippet': 'The <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span> (Punjabi: شاہی قلعہ, romanised:\xa0Śā&#039;ī Qilā; Urdu: شاہی قلعہ, romanised:\xa0Shahi Qilah; lit.\u2009&#039;Royal <span class="searchmatch">Fort</span>&#039;) is a citadel in the walled interior'}, {'title': 'Sheesh Mahal (Lahore Fort)', 'snippet': 'located within the Shah Burj block at the north-western corner of the <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span>, in <span class="searchmatch">Lahore</span>, Pakistan. It was constructed during the reign of Mughal Emperor'}, {'title': 'Lahore', 'snippet': 'shrines. <span class="searchmatch">Lahore</span> is also home to the <span class="searchmatch">Lahore</span> <span class="searchmatch">Fort</span> and Shalimar Gardens, both of which are UNESCO World Heritage Sites. The origin of <span class="searchmatch">Lahore&#039;s</span> name is unclear'}]}


Now write the schema (same pattern as `get_weather_declaration`), register it, and test the
agent end to end with a query that needs Wikipedia.


In [16]:
# Declaration → Gemini knows the tool
search_wikipedia_declaration = types.FunctionDeclaration(
    name="search_wikipedia",
    description="Search Wikipedia for information about a topic and return relevant search results.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The topic or question to search for on Wikipedia."
            },
            "num_results": {
                "type": "integer",
                "description": "The maximum number of Wikipedia results to return."
            }
        },
        "required": ["query"]
    }
)

In [17]:
# Registry → Python knows which function to execute
multi_tool = types.Tool(
    function_declarations=[
        get_weather_declaration,
        search_wikipedia_declaration
    ]
)

config = types.GenerateContentConfig(
    tools=[multi_tool]
)

tool_registry = {
    "get_weather": get_weather,
    "search_wikipedia": search_wikipedia
}

**🔧 Checkpoint (3 min):** ask a question that should trigger a zero-result Wikipedia search
(a nonsense query) and confirm your agent handles an empty `results` list without crashing.


In [19]:
# Now give it a nonsense query, for example:
answer = run_agent_v2(
    "Search Wikipedia for Xyzzyqwertyuniverse123456789.",
    config=config,
    tool_registry=tool_registry
)

print(answer)
# The important thing is that your function should return:
# {
#   "results": []
#} rather than crashing.

[Step 1] Action: search_wikipedia({'query': 'Xyzzyqwertyuniverse123456789'})
[Step 1] Observation: {'results': []}
[Step 2] Thought: I have enough info. Giving final answer.
A search on Wikipedia for **Xyzzyqwertyuniverse123456789** returned no results.


---
## Section 5 — Write a Real Database Tool + Test Full Chaining

Real production agents often query a real database. We'll create a small local SQLite
database (given — creating sample data isn't the learning goal here) and you'll write the
**query tool** yourself.


In [22]:
import sqlite3

conn = sqlite3.connect("orders.db")

conn.execute("DROP TABLE IF EXISTS orders")

conn.execute("""
    CREATE TABLE orders (
        order_id TEXT,
        customer_id TEXT,
        status TEXT
    )
""")

sample_orders = [
    ("ORD-001", "CUST123", "shipped"),
    ("ORD-002", "CUST123", "processing"),
    ("ORD-003", "CUST456", "delivered"),
]

conn.executemany(
    "INSERT INTO orders VALUES (?, ?, ?)",
    sample_orders
)

conn.commit()
conn.close()

print("orders.db ready")

orders.db ready


**Spec for the tool:**
- Signature: `query_orders(customer_id: str) -> dict`
- Connect to `orders.db`, run a **parameterized** query (use `?` placeholders — never
  string-format `customer_id` directly into SQL) selecting `order_id, status` for that customer
- Return `{"orders": [{"order_id": ..., "status": ...}, ...]}`
- If there are no matching rows, return `{"orders": []}` (valid, not an error)


In [23]:
query_orders_declaration = types.FunctionDeclaration(
    name="query_orders",
    description="Look up the orders and their current status for a specific customer.",
    parameters={
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "The customer ID, for example CUST123."
            }
        },
        "required": ["customer_id"]
    }
)

In [24]:
all_tools = types.Tool(
    function_declarations=[
        get_weather_declaration,
        search_wikipedia_declaration,
        query_orders_declaration
    ]
)

In [25]:
config = types.GenerateContentConfig(
    tools=[all_tools]
)

In [26]:
tool_registry = {
    "get_weather": get_weather,
    "search_wikipedia": search_wikipedia,
    "query_orders": query_orders
}

### Test full 3-tool chaining


In [27]:
answer = run_agent_v2(
    "What's the weather in Lahore, tell me one fact about Lahore from Wikipedia, "
    "and check the order status for customer CUST123.",
    config=config,
    tool_registry=tool_registry
)

print()
print("=" * 50)
print("FINAL ANSWER:", answer)

[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 33.0, 'windspeed': 1.4}
[Step 1] Action: search_wikipedia({'query': 'Lahore', 'num_results': 1})
[Step 1] Observation: {'results': [{'title': 'Lahore', 'snippet': '<span class="searchmatch">Lahore</span> is the capital and largest city of the Pakistani province of Punjab. It is the second-largest city in Pakistan, after Karachi, and 27th largest'}]}
[Step 1] Action: query_orders({'customer_id': 'CUST123'})
[Step 1] Observation: {'orders': [{'order_id': 'ORD-001', 'status': 'shipped'}, {'order_id': 'ORD-002', 'status': 'processing'}]}
[Step 2] Thought: I have enough info. Giving final answer.

FINAL ANSWER: Here is the information you requested:

1. **Weather in Lahore:** 
   - **Temperature:** 33°C
   - **Wind Speed:** 1.4 km/h

2. **Fact from Wikipedia:** 
   - Lahore is the capital and largest city of the Pakistani province of Punjab, as well as the second-largest city in Pakistan (aft

---
## Section 6 — Independent Challenge: Break It, Then Prove It Survives

You already wrapped `get_weather` in resilience via `run_agent_v2` — that wrapping applies
automatically to *every* tool in the registry, including the two you just wrote. Prove it.

**Task:** deliberately trigger a real failure in each tool and confirm the agent still returns
a sensible final answer instead of crashing.


In [28]:
# Breaking each tool on purpose, one at a time, and confirm graceful handling:

# 1. A city that doesn't exist
print(
    run_agent_v2(
        "Weather in Xyzzyqwertyville999?",
        config=config,
        tool_registry=tool_registry,
        verbose=False
    )
)

# 2. A customer ID that doesn't exist
print(
    run_agent_v2(
        "Order status for customer NOBODY999?",
        config=config,
        tool_registry=tool_registry,
        verbose=False
    )
)

# 3. A Wikipedia query that returns nothing
print(
    run_agent_v2(
        "Wikipedia fact about Xyzzyqwertyuniverse123456789?",
        config=config,
        tool_registry=tool_registry,
        verbose=False
    )
)


I couldn't retrieve weather data for "Xyzzyqwertyville999" because no location matching that name was found. Please check the spelling or provide a valid city name.
No orders were found for customer ID **NOBODY999**. Please verify the customer ID and try again.
No Wikipedia articles or facts were found for **Xyzzyqwertyuniverse123456789**. It appears to be a fictional or randomized name that does not exist in Wikipedia's database.


---
## Section 7 — Iteration Log Analysis
This is today's core lab task from the slides: **reading a trace and diagnosing a failure.**

First, generate your own real broken trace:


#**Important Note:**
In the following cell, the error specifically asks me  to retry after about 35 seconds, although if I hit the daily/project quota, waiting 35 seconds may not be enough. My error is saying that I exceeded my current quota, check your plan and billing details.

> So, ignore this error message kindly for the below cell.



In [31]:
# GIVEN — run with verbose=True so every step prints, then deliberately confuse it
trace_answer = run_agent_v2(
    "What's the weather in Lahoreee and check order status for CUST999?",  # typo'd city + fake customer
    config=config, tool_registry=tool_registry, verbose=True
)
print()
print("FINAL ANSWER:", trace_answer) # if after runnig the cell, there is an error so kindly read the above markdown cell


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 40.950101769s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

Now analyze this **pre-recorded** trace from another (deliberately broken) run — a teammate's
agent that got stuck:

```
[Step 1] Action: get_weather({'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'temp_c': 34, 'windspeed': 8.2}
[Step 2] Action: query_orders({'customer_id': 'CUST-123'})
[Step 2] Observation: {'orders': []}
[Step 3] Action: query_orders({'customer_id': 'CUST-123'})
[Step 3] Observation: {'orders': []}
[Step 4] Action: query_orders({'customer_id': 'CUST-123'})
[Step 4] Observation: {'orders': []}
Reached max steps without a final answer.
```

### Answer these in the markdown cell below (double-click to edit)
1. The weather lookup worked fine. What went wrong with the orders lookup, specifically?
   (Hint: compare `'CUST-123'` here against the sample data format from Section 5.)
2. Why did the agent call `query_orders` three times with the *exact same* (wrong) argument
   instead of trying something different or giving up sooner?
3. Whose fault is this, really — the model's, the tool's, or the data's? What's the one-line
   fix?


1. The weather lookup worked fine, but the orders lookup used the wrong customer ID format. The sample data from Section 5 uses `CUST123`, while the agent passed `CUST-123`. Because of the extra hyphen, the database could not find any matching orders and returned `{"orders": []}`.

2. The agent called `query_orders` three times because the ReAct loop continued after receiving the empty result. The model did not change the argument or decide to give a final answer, so it repeated the same tool call. The `max_steps` limit eventually stopped the loop and prevented it from running forever.

3. The main fault was the model generating the wrong customer ID argument. The `query_orders` tool and database worked correctly because they searched for exactly what they were given. The one-line fix is: use `CUST123` instead of `CUST-123`.

 # **Your analysis:**

1. The weather lookup worked correctly, but the order lookup failed because the model used CUST-123 instead of the correct customer ID format CUST123.

2. The agent repeated the same query_orders call because the ReAct loop continued after receiving an empty result, and the model did not change its argument or provide a final answer. The max_steps limit eventually stopped the loop.

3. The main fault was the model generating the wrong customer ID. The tool and database worked correctly. The one-line fix is to use CUST123 instead of CUST-123.



